### Ingestion del archivo "country.json"

In [0]:
dbutils.widgets.text("p_environment", "production")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

#### Paso 1 - Leer el archivo JSON usando "DataFrameReader" de spark

In [0]:
countries_schema = "countryId INT, countryIsoCode String, countryName String"

In [0]:
countries_df = spark.read \
               .schema(countries_schema) \
               .json(f"{bronze_folder_path}/{v_file_date}/country.json")

In [0]:
countries_df.printSchema()

root
 |-- countryId: integer (nullable = true)
 |-- countryIsoCode: string (nullable = true)
 |-- countryName: string (nullable = true)



#### Paso 2 - Eliminar columnas no deseadas del DataFrame

In [0]:
from pyspark.sql.functions import col

In [0]:
countries_dropped_df = countries_df.drop(col("countryIsoCode"))

#### Paso 3 - Cambiar el nombre de las columnas y añadir "ingestion_date" y "environment"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
countries_final_df = add_ingestion_date(countries_dropped_df) \
                     .withColumnRenamed("countryId", "country_Id") \
                     .withColumnRenamed("countryName", "country_Name") \
                     .withColumn("environment", lit(v_environment)) \
                     .withColumn("file_date", lit(v_file_date))

#### Paso 4 - Escribir la salida en un formato "Parquet"

In [0]:
countries_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.countries")

In [0]:
display(countries_final_df)

country_Id,country_Name,ingestion_date,environment,file_date
128,United Arab Emirates,2026-09-10T20:10:12.766451Z,production,2024-12-30
129,Afghanistan,2026-09-10T20:10:12.766451Z,production,2024-12-30
130,Angola,2026-09-10T20:10:12.766451Z,production,2024-12-30
131,Argentina,2026-09-10T20:10:12.766451Z,production,2024-12-30
132,Austria,2026-09-10T20:10:12.766451Z,production,2024-12-30
133,Australia,2026-09-10T20:10:12.766451Z,production,2024-12-30
134,Aruba,2026-09-10T20:10:12.766451Z,production,2024-12-30
135,Bosnia and Herzegovina,2026-09-10T20:10:12.766451Z,production,2024-12-30
136,Belgium,2026-09-10T20:10:12.766451Z,production,2024-12-30
137,Bulgaria,2026-09-10T20:10:12.766451Z,production,2024-12-30


In [0]:
display(spark.read.table("movie_silver.countries"))

country_Id,country_Name,ingestion_date,environment,file_date
128,United Arab Emirates,2026-09-10T20:10:11.108559Z,production,2024-12-30
129,Afghanistan,2026-09-10T20:10:11.108559Z,production,2024-12-30
130,Angola,2026-09-10T20:10:11.108559Z,production,2024-12-30
131,Argentina,2026-09-10T20:10:11.108559Z,production,2024-12-30
132,Austria,2026-09-10T20:10:11.108559Z,production,2024-12-30
133,Australia,2026-09-10T20:10:11.108559Z,production,2024-12-30
134,Aruba,2026-09-10T20:10:11.108559Z,production,2024-12-30
135,Bosnia and Herzegovina,2026-09-10T20:10:11.108559Z,production,2024-12-30
136,Belgium,2026-09-10T20:10:11.108559Z,production,2024-12-30
137,Bulgaria,2026-09-10T20:10:11.108559Z,production,2024-12-30


In [0]:
%sql
SELECT * FROM movie_silver.countries

country_Id,country_Name,ingestion_date,environment,file_date
128,United Arab Emirates,2026-09-10T20:10:11.108559Z,production,2024-12-30
129,Afghanistan,2026-09-10T20:10:11.108559Z,production,2024-12-30
130,Angola,2026-09-10T20:10:11.108559Z,production,2024-12-30
131,Argentina,2026-09-10T20:10:11.108559Z,production,2024-12-30
132,Austria,2026-09-10T20:10:11.108559Z,production,2024-12-30
133,Australia,2026-09-10T20:10:11.108559Z,production,2024-12-30
134,Aruba,2026-09-10T20:10:11.108559Z,production,2024-12-30
135,Bosnia and Herzegovina,2026-09-10T20:10:11.108559Z,production,2024-12-30
136,Belgium,2026-09-10T20:10:11.108559Z,production,2024-12-30
137,Bulgaria,2026-09-10T20:10:11.108559Z,production,2024-12-30


In [0]:
dbutils.notebook.exit("Success")